# Leitura de dados brutos, CSV e JSON

Neste notebook, vamos conhecer diferentes formas de trazer dados
- lendo bytes e texto diretamente
- interpretando arquivos CSV e json com a biblioteca padrão e usando **pandas**

## Objetivos
- Diferenciar a leitura de bytes da interpretação dos dados.
- Ler um arquivo texto e um CSV com ferramentas da biblioteca padrão.
- Ler CSV e JSON com pandas e inspecionar os dados.
- Reconhecer problemas comuns de separador, codificação, tipos e dados ausentes.
- Fazer requisições HTTP, percorrer páginas e tratar indisponibilidade e conflitos de API.
- Extrair e relacionar dados de um banco OLTP local usando SQLite.

### Observações
- Os exemplos de arquivos e do banco SQLite serão salvos na pasta `dados_aula`.
- A API REST é simulada localmente, sem depender de serviços externos.
- Execute cada bloco de código em sequência para que tudo funcione.

## 1. Preparação

`pathlib` facilita a criação de caminhos de arquivos. `csv` e `json` fazem parte da biblioteca padrão do Python, portanto não precisam ser instalados. O pandas será importado mais adiante.

In [1]:
from pathlib import Path
import csv
import json

pasta_dados = Path("dados_aula")
pasta_dados.mkdir(exist_ok=True)
print(f"Arquivos de demonstração serão salvos em: {pasta_dados.resolve()}")

Arquivos de demonstração serão salvos em: C:\Users\ajb95\Faculdade\LADATA\SEMAC\testes\dados_aula


## 2. Leitura de dados brutos

“Bruto” significa que ainda não interpretamos o conteúdo como uma tabela ou estrutura específica. Arquivos podem ser lidos como **texto** (caracteres) ou como **bytes** (valores binários). Texto exige atenção à codificação, como UTF-8. Bytes são úteis para arquivos binários, como imagens, áudio e formatos compactados.

In [2]:
arquivo_texto = pasta_dados / "anotacoes.txt"
arquivo_texto.write_text("Amostra de dados brutos.\nTemperatura: 23.5 °C\n", encoding="utf-8")

# Leitura como texto: o arquivo é aberto para leitura e tem os bytes decodificados usando UTF-8.
with arquivo_texto.open("r", encoding="utf-8") as arquivo:
    conteudo_texto = arquivo.read()
print(conteudo_texto)

# Leitura binária: recebemos bytes, sem interpretação automática como texto.
conteudo_bytes = arquivo_texto.read_bytes()
print("Tipo:", type(conteudo_bytes))
print("Primeiros bytes:", conteudo_bytes[:30])
print("Bytes em hexadecimal:", conteudo_bytes[:12].hex())

Amostra de dados brutos.
Temperatura: 23.5 °C

Tipo: <class 'bytes'>
Primeiros bytes: b'Amostra de dados brutos.\r\nTemp'
Bytes em hexadecimal: 416d6f737472612064652064


O mesmo arquivo pode ser visto como texto ou como bytes. Ao ler bytes de um arquivo que sabemos ser texto, podemos decodificá-los explicitamente com `.decode("utf-8")`. Para dados binários, não devemos tentar decodificar sem saber o formato.

In [3]:
texto_decodificado = conteudo_bytes.decode("utf-8")
print(texto_decodificado)

Amostra de dados brutos.
Temperatura: 23.5 °C



## 3. CSV com a biblioteca padrão (`csv`)

CSV (comma separated values) representa linhas e colunas em texto separado por um delimitador. Embora a vírgula seja comum, arquivos também podem usar ponto e vírgula. `csv.DictReader` transforma cada linha em um dicionário cujas chaves são os nomes das colunas. Sem conversão adicional, os valores lidos são textos.

In [4]:
#guardando valores num arquivo csv
arquivo_csv = pasta_dados / "alunos.csv" # cria o arquivo na pasta
linhas_alunos = [
    {"nome": "Ana", "idade": 20, "nota": 8.5},
    {"nome": "Bruno", "idade": 22, "nota": 7.0},
    {"nome": "Caio", "idade": 19, "nota": 9.2},
] # dicionario com valores representado nome, idade e nota, que serão o cabeçalho da nossa tabela

with arquivo_csv.open("w", newline="", encoding="utf-8") as arquivo: #abertura do arquivo para escrita
    escritor = csv.DictWriter(arquivo, fieldnames=["nome", "idade", "nota"]) #declarando os nomes das colunas (cabeçalho)
    escritor.writeheader() # escrita do cabeçalho no csv
    escritor.writerows(linhas_alunos) #escrita de cada linha e seus valores nas respectivas colunas

#leitura do que foi escrito no arquivo
with arquivo_csv.open("r", newline="", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    alunos = list(leitor) # guarda o conteudo na lista de alunos

print(alunos)
print("Tipo da idade lida:", type(alunos[0]["idade"]))

[{'nome': 'Ana', 'idade': '20', 'nota': '8.5'}, {'nome': 'Bruno', 'idade': '22', 'nota': '7.0'}, {'nome': 'Caio', 'idade': '19', 'nota': '9.2'}]
Tipo da idade lida: <class 'str'>


## 4. CSV com pandas

O pandas representa dados tabulares em um `DataFrame`, facilitando filtros, conversões, estatísticas e exploração. Se o import abaixo falhar porque pandas não está instalado, adicione uma célula de código e execute: `%pip install pandas`, depois, execute novamente o import.

In [5]:
import pandas as pd

print("Versão do pandas:", pd.__version__)

Versão do pandas: 2.3.3


A função `read_csv` lê o arquivo e tenta inferir os tipos das colunas. Vamos criar uma tabela com um valor ausente e explorar suas primeiras linhas, tipos e estatísticas.

In [6]:
#criando valores e gravando no arquivo vendas.csv
arquivo_vendas = pasta_dados / "vendas.csv"
vendas_exemplo = pd.DataFrame(
    {
        "produto": ["Café", "Chá", "Café", "Suco"],
        "quantidade": [3, 5, 2, 4],
        "preco_unitario": [12.5, 8.0, 12.5, 6.5],
        "loja": ["Centro", "Norte", "Centro", None],
    }
)
#index=false indica que nesse caso nao queremos que sej agravada uma coluna adicional com o indice de cada linha
vendas_exemplo.to_csv(arquivo_vendas, index=False, encoding="utf-8") 

#lendo o arquivo com pandas
vendas = pd.read_csv(arquivo_vendas, encoding="utf-8")

# head mostra as primeiras linhas do arquivo
display(vendas.head()) #
print("\nResumo das colunas e tipos:")
vendas.info()
print("\nEstatísticas numéricas:")
display(vendas.describe())
print("\nValores ausentes por coluna:")
display(vendas.isna().sum())

,produto,quantidade,preco_unitario,loja
0,Café,3,12.5,Centro
1,Chá,5,8.0,Norte
2,Café,2,12.5,Centro
3,Suco,4,6.5,NaN



Resumo das colunas e tipos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   produto         4 non-null      object 
 1   quantidade      4 non-null      int64  
 2   preco_unitario  4 non-null      float64
 3   loja            3 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 260.0+ bytes

Estatísticas numéricas:


,quantidade,preco_unitario
count,4.000000,4.000000
mean,3.500000,9.875000
std,1.290994,3.092329
min,2.000000,6.500000
25%,2.750000,7.625000
50%,3.500000,10.250000
75%,4.250000,12.500000
max,5.000000,12.500000



Valores ausentes por coluna:


produto           0
quantidade        0
preco_unitario    0
loja              1
dtype: int64

### Opções úteis ao ler CSV

- `sep`: delimitador, por exemplo `sep=";"`.
- `encoding`: codificação do arquivo, frequentemente `"utf-8"` ou `"latin-1"`.
- `usecols`: seleciona apenas algumas colunas.
- `dtype`: define tipos quando a inferência automática não é adequada.
- `na_values`: informa textos que devem ser tratados como valores ausentes.

Um separador ou encoding incorreto pode fazer a leitura falhar ou produzir colunas inesperadas, por isso é importante conhecer como o arquivo foi gerado e sua estrutura

In [7]:
arquivo_ponto_virgula = pasta_dados / "estoque.csv"
arquivo_ponto_virgula.write_text(
    "item;quantidade\nCaderno;10\nCaneta;25\n", encoding="utf-8"
)

estoque = pd.read_csv(arquivo_ponto_virgula, sep=";", encoding="utf-8")
display(estoque)
print("Tipos inferidos:", estoque.dtypes.to_dict())

,item,quantidade
0,Caderno,10
1,Caneta,25


Tipos inferidos: {'item': dtype('O'), 'quantidade': dtype('int64')}


## 5. JSON com a biblioteca padrão (`json`)

JSON é um formato textual que representa objetos, listas, números, textos, valores booleanos e `null`. Em Python, `json.load` lê JSON de um arquivo aberto; `json.loads` interpreta uma string. O resultado costuma ser um dicionário ou uma lista.

In [8]:
arquivo_json = pasta_dados / "aluno.json"
aluno = {
    "nome": "Lia",
    "curso": "Ciência de Dados",
    "ativo": True,
    "notas": [8.5, 9.0],
}

with arquivo_json.open("w", encoding="utf-8") as arquivo:
    json.dump(aluno, arquivo, ensure_ascii=False, indent=2)

with arquivo_json.open("r", encoding="utf-8") as arquivo:
    aluno_lido = json.load(arquivo)

print(aluno_lido)
print("Nome:", aluno_lido["nome"])
print("Média das notas:", sum(aluno_lido["notas"]) / len(aluno_lido["notas"]))

# Também podemos interpretar JSON que já está em uma string.
texto_json = '[{"cidade": "Recife", "temperatura": 28}, {"cidade": "Curitiba", "temperatura": 17}]'
cidades = json.loads(texto_json)
print("\nLista interpretada a partir de uma string:", cidades)

{'nome': 'Lia', 'curso': 'Ciência de Dados', 'ativo': True, 'notas': [8.5, 9.0]}
Nome: Lia
Média das notas: 8.75

Lista interpretada a partir de uma string: [{'cidade': 'Recife', 'temperatura': 28}, {'cidade': 'Curitiba', 'temperatura': 17}]


## 6. JSON com pandas

Para JSON organizado como uma lista de objetos com campos semelhantes, `pd.read_json` pode criar um `DataFrame` diretamente. Para respostas de APIs ou estruturas aninhadas, `pd.json_normalize` ajuda a transformar campos internos em colunas.

In [9]:
arquivo_cidades = pasta_dados / "cidades.json"
registros_cidades = [
    {"cidade": "Recife", "temperatura": 28, "estado": "PE"},
    {"cidade": "Curitiba", "temperatura": 17, "estado": "PR"},
    {"cidade": "Manaus", "temperatura": 31, "estado": "AM"},
]
with arquivo_cidades.open("w", encoding="utf-8") as arquivo:
    json.dump(registros_cidades, arquivo, ensure_ascii=False, indent=2)

cidades_df = pd.read_json(arquivo_cidades, encoding="utf-8")
display(cidades_df)
print("Temperatura média:", cidades_df["temperatura"].mean())

,cidade,temperatura,estado
0,Recife,28,PE
1,Curitiba,17,PR
2,Manaus,31,AM


Temperatura média: 25.333333333333332


In [10]:
resposta_api = {
    "fonte": "estacao_local",
    "leituras": [
        {"local": {"cidade": "Recife", "estado": "PE"}, "medicao": {"temperatura": 28, "umidade": 70}},
        {"local": {"cidade": "Curitiba", "estado": "PR"}, "medicao": {"temperatura": 17, "umidade": 55}},
    ],
}

# record_path escolhe a lista de registros; meta preserva um campo do objeto externo.
leituras_df = pd.json_normalize(
    resposta_api,
    record_path="leituras",
    meta=["fonte"],
    sep="_",
)
display(leituras_df)

,local_cidade,local_estado,medicao_temperatura,medicao_umidade,fonte
0,Recife,PE,28,70,estacao_local
1,Curitiba,PR,17,55,estacao_local


## 7. Qual ferramenta escolher?

| Situação | Ferramenta indicada |
| --- | --- |
| Ler bytes de um arquivo ou texto simples | `Path.read_bytes()` / `open()` |
| Processar CSV pequeno sem dependências extras | `csv.DictReader` |
| Explorar, filtrar e analisar dados tabulares | `pandas.read_csv` |
| Interpretar JSON e preservar sua estrutura | `json.load` / `json.loads` |
| Converter JSON tabular ou aninhado em colunas | `pandas.read_json` / `pandas.json_normalize` |

### Boas práticas
1. Confira a codificação e o delimitador do arquivo.
2. Inspecione as primeiras linhas e os tipos antes de analisar.
3. Verifique valores ausentes e converta tipos quando necessário.
4. Use `with open(...)` para garantir o fechamento dos arquivos.
5. Para arquivos grandes, considere processar em partes (por exemplo, `chunksize` no `read_csv`).


**Observação:** os arquivos de exemplo são recriados quando as células correspondentes são executadas. Execute as células de cima para baixo para acompanhar a aula.

## 9. Ingestão via API REST

Uma API REST funciona como uma ponte de comunicação para extrair dados da web. Em um processo clássico de ingestão, o engenheiro de dados faz um pedido à API (requisição GET), verifica se o status da resposta foi um sucesso, interpreta os dados recebidos (geralmente em formato JSON) e percorre as páginas seguintes até extrair tudo o que precisa.
No mundo corporativo, essas APIs costumam exigir autenticação, impor limites de acesso rápido e usar regras próprias de paginação. Então, para simularmos esses desafios reais de forma prática, vamos consumir uma API pública de verdade: a PokeAPI. 

- https://pokeapi.co/

No código a seguir, vamos demontrar passo a passo como se dá todo esse processo.


## 10. Ingestão de banco de dados relacional (OLTP)

Bancos OLTP (*Online Transaction Processing*) são voltados a operações do dia a dia, como cadastrar clientes e registrar pedidos. Os dados costumam estar organizados em tabelas relacionadas. Para simular a extração sem instalar um servidor, usaremos o SQLite local e o módulo `sqlite3`, incluído no Python.

In [15]:
import sqlite3

arquivo_banco = pasta_dados / "loja_oltp.db"
conexao = sqlite3.connect(arquivo_banco)

try:
    # Cria duas tabelas relacionadas.
    conexao.execute("""
        CREATE TABLE clientes (
            cliente_id INTEGER PRIMARY KEY,
            nome TEXT NOT NULL,
            cidade TEXT NOT NULL
        )
    """)

    conexao.execute("""
        CREATE TABLE pedidos (
            pedido_id INTEGER PRIMARY KEY,
            cliente_id INTEGER NOT NULL,
            data_pedido TEXT NOT NULL,
            valor REAL NOT NULL,
            FOREIGN KEY (cliente_id) REFERENCES clientes(cliente_id)
        )
    """)

    # Insere alguns registros fictícios.
    conexao.executemany(
        "INSERT INTO clientes (cliente_id, nome, cidade) VALUES (?, ?, ?)",
        [
            (1, "Ana", "Recife"),
            (2, "Bruno", "Curitiba"),
            (3, "Lia", "Manaus"),
        ],
    )

    conexao.executemany(
        """INSERT INTO pedidos
           (pedido_id, cliente_id, data_pedido, valor)
           VALUES (?, ?, ?, ?)""",
        [
            (1001, 1, "2026-03-01", 45.90),
            (1002, 2, "2026-03-02", 18.00),
            (1003, 1, "2026-03-03", 12.50),
            (1004, 3, "2026-03-04", 75.00),
            (1005, 2, "2026-03-05", 32.25),
        ],
    )

    conexao.commit()

finally:
    conexao.close()

print("Banco de demonstração criado em:", arquivo_banco)

Banco de demonstração criado em: dados_aula\loja_oltp.db


### Conectar e executar uma consulta
A função sqlite3.connect() abre a conexão com o arquivo do banco. Com cursor.execute(), executamos uma instrução SQL. O método fetchall() recupera todas as linhas retornadas pela consulta.

Neste primeiro exemplo, vamos consultar os pedidos sem relacioná-los a outras tabelas.

In [23]:
conexao = sqlite3.connect(arquivo_banco)

try:
    cursor = conexao.cursor()

    cursor.execute("""
        SELECT pedido_id, data_pedido, valor
        FROM pedidos
        ORDER BY pedido_id
    """)

    registros = cursor.fetchall()

    for registro in registros:
        print(registro)

finally:
    conexao.close()

(1001, '2026-03-01', 45.9)
(1002, '2026-03-02', 18.0)
(1003, '2026-03-03', 12.5)
(1004, '2026-03-04', 75.0)
(1005, '2026-03-05', 32.25)


O resultado de fetchall() é uma lista de tuplas. Quando não precisamos de todos os registros, podemos recuperar apenas um (fetchone()) ou uma quantidade limitada (fetchmany()).

Também podemos filtrar os dados diretamente no SQL. Para isso, utilizamos parâmetros (?) em vez de inserir valores diretamente no texto da consulta. Essa prática evita problemas com valores especiais e reduz riscos de injeção de SQL.

In [25]:
conexao = sqlite3.connect(arquivo_banco)

try:
    cursor = conexao.cursor()

    consulta = """
        SELECT pedido_id, data_pedido, valor
        FROM pedidos
        WHERE valor >= ?
        ORDER BY valor DESC
    """

    valor_minimo = 30.00
    cursor.execute(consulta, (valor_minimo,))

    print("Pedidos com valor igual ou superior a R$ 30:")
    for pedido in cursor.fetchall():
        print(pedido)

finally:
    conexao.close()

Pedidos com valor igual ou superior a R$ 30:
(1004, '2026-03-04', 75.0)
(1001, '2026-03-01', 45.9)
(1005, '2026-03-05', 32.25)


Em bancos relacionais, uma informação pode estar distribuída em diferentes tabelas. O JOIN permite combinar registros relacionados.

No exemplo, a tabela pedidos armazena o identificador do cliente, enquanto o nome e a cidade estão na tabela clientes. A consulta reúne essas informações em um único resultado.

In [26]:
conexao = sqlite3.connect(arquivo_banco)

try:
    consulta = """
        SELECT
            pedidos.pedido_id,
            pedidos.data_pedido,
            clientes.nome AS cliente,
            clientes.cidade,
            pedidos.valor
        FROM pedidos
        JOIN clientes
            ON pedidos.cliente_id = clientes.cliente_id
        ORDER BY pedidos.pedido_id
    """

    cursor = conexao.cursor()
    cursor.execute(consulta)

    for pedido in cursor.fetchall():
        print(pedido)

finally:
    conexao.close()

(1001, '2026-03-01', 'Ana', 'Recife', 45.9)
(1002, '2026-03-02', 'Bruno', 'Curitiba', 18.0)
(1003, '2026-03-03', 'Ana', 'Recife', 12.5)
(1004, '2026-03-04', 'Lia', 'Manaus', 75.0)
(1005, '2026-03-05', 'Bruno', 'Curitiba', 32.25)


Além de recuperar os registros com sqlite3, podemos utilizar o pandas para executar uma consulta e receber o resultado diretamente em um DataFrame.

Essa integração é útil quando os dados extraídos serão explorados, analisados ou encaminhados para outras etapas do fluxo de engenharia de dados.

In [27]:
conexao = sqlite3.connect(arquivo_banco)

try:
    pedidos_df = pd.read_sql_query(consulta, conexao)
    display(pedidos_df)

finally:
    conexao.close()

,pedido_id,data_pedido,cliente,cidade,valor
0,1001,2026-03-01,Ana,Recife,45.90
1,1002,2026-03-02,Bruno,Curitiba,18.00
2,1003,2026-03-03,Ana,Recife,12.50
3,1004,2026-03-04,Lia,Manaus,75.00
4,1005,2026-03-05,Bruno,Curitiba,32.25


### Extrair dados em lotes
Quando uma consulta retorna muitos registros, carregar tudo de uma vez pode consumir muita memória. O método fetchmany(tamanho) permite recuperar os resultados em grupos menores.

Neste exemplo, cada chamada recupera até dois registros. Em um fluxo real, os lotes poderiam ser enviados para arquivos, outro banco de dados ou uma etapa de processamento.

In [28]:
conexao = sqlite3.connect(arquivo_banco)

try:
    cursor = conexao.cursor()
    cursor.execute(consulta)

    tamanho_lote = 2
    numero_lote = 1

    while True:
        lote = cursor.fetchmany(tamanho_lote)

        if not lote:
            break

        print(f"Lote {numero_lote}:")
        for pedido in lote:
            print(pedido)

        numero_lote += 1

finally:
    conexao.close()

Lote 1:
(1001, '2026-03-01', 'Ana', 'Recife', 45.9)
(1002, '2026-03-02', 'Bruno', 'Curitiba', 18.0)
Lote 2:
(1003, '2026-03-03', 'Ana', 'Recife', 12.5)
(1004, '2026-03-04', 'Lia', 'Manaus', 75.0)
Lote 3:
(1005, '2026-03-05', 'Bruno', 'Curitiba', 32.25)


### Cuidados ao extrair dados relacionais

- Conheça a estrutura: identifique tabelas, colunas, chaves e relacionamentos antes de escrever consultas.
- Selecione o necessário: evite trazer todas as colunas e todos os registros quando isso não for necessário.
- Use parâmetros: passe valores por meio de ?, sem concatenar dados diretamente no comando SQL.
- Considere o volume: fetchmany() permite ler o resultado em lotes; para volumes maiores, também podem ser necessárias estratégias de paginação ou extração incremental.
- Evite sobrecarregar a origem: consultas muito longas ou frequentes podem afetar o banco utilizado pelos sistemas operacionais.
- Feche a conexão: utilize try/finally ou outra estrutura que garanta o fechamento após a extração.
